# 14 - Performance

Vectorised NumPy is fast because it pushes the work into compiled C loops. To keep it fast:

### Memory layout

- **C-order (row-major)**: rows stored contiguously — `np.array(a, order="C")`.
- **F-order (column-major)**: columns stored contiguously — `order="F"`.
- Access **along the contiguous axis** is fastest (`a[i, :]` for C-order, `a[:, j]` for F-order).
- `np.ascontiguousarray(a)` forces C layout; `.strides` shows bytes between consecutive elements.

### Views vs copies

- **Views** share memory (`reshape`, `transpose`, slicing) — cheap but mutations leak.
- **Copies** (`a.copy()`, `.astype()`) are independent.
- Check with `np.shares_memory(a, b)`.

### Avoid Python loops

- Use ufuncs and reductions instead of `for` loops.
- `np.fromiter` builds arrays without list round-trips.
- `np.ufunc.at(a, idx, vals)` applies a ufunc at specific positions (e.g. repeated-index accumulation).
- `np.einsum` fuses multi-axis products like `matmul`, trace, and diagonal sums in one pass.

### Smaller dtype = less memory

- `int32`/`float32` halve memory vs `int64`/`float64`.
- `a.astype("float32")` and `np.memmap(..., dtype="float32")`.

In [1]:
import numpy as np
import time

## 1. C-order vs F-order — strides

Strides = bytes to step to the next element along each axis.

In [2]:
C = np.array(np.arange(24).reshape(4, 6), order="C")
F = np.array(np.arange(24).reshape(4, 6), order="F")
print("C-order strides:", C.strides)
print("F-order strides:", F.strides)
print("C is contiguous:", C.flags.c_contiguous, "| F contiguous:", C.flags.f_contiguous)

C-order strides: (24, 4)
F-order strides: (4, 16)
C is contiguous: True | F contiguous: False


In [3]:
# Reading rows (contiguous in C) is faster than columns
def row_mean(a):
    return np.mean([row.mean() for row in a])

bigC = np.random.default_rng(0).random((500, 500))
t0 = time.perf_counter(); row_mean(bigC); tC = time.perf_counter() - t0

bigF = np.array(bigC, order="F")
t0 = time.perf_counter(); row_mean(bigF); tF = time.perf_counter() - t0
print(f"C-order row loop: {tC*1000:.2f} ms | F-order row loop: {tF*1000:.2f} ms")

C-order row loop: 13.85 ms | F-order row loop: 19.41 ms


## 2. Views vs copies

Slices and `reshape` give views (shared memory); be careful mutating them.

In [4]:
x = np.arange(10)
view = x[2:5]          # a VIEW
copy = x[2:5].copy()  # a COPY
view[:] = -1
print("after mutating the view:")
print("x   :", x)
print("copy:", copy)
print("shares memory with x:", np.shares_memory(x, view), np.shares_memory(x, copy))

after mutating the view:
x   : [ 0  1 -1 -1 -1  5  6  7  8  9]
copy: [2 3 4]
shares memory with x: True False


## 3. Vectorisation beats loops

A Python loop over a million elements is far slower than a ufunc.

In [5]:
big = np.random.default_rng(1).random(1_000_000)

t0 = time.perf_counter()
sq = big * big
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
lst = [v * v for v in big]
t_loop = time.perf_counter() - t0

print(f"vectorised: {t_vec*1000:.2f} ms | python loop: {t_loop*1000:.2f} ms | speedup {t_loop/t_vec:.0f}x")

vectorised: 7.41 ms | python loop: 745.62 ms | speedup 101x


## 4. `fromiter` and `ufunc.at`

`fromiter` builds arrays from generators without list round-trips; `ufunc.at` handles repeated indices.

In [6]:
arr = np.fromiter((i * i for i in range(10)), dtype=int)
arr

array([ 0,  1,  4,  9, 16, 25, 36, 49, 64, 81])

In [7]:
# Accumulate additions at repeated positions
out = np.zeros(5, dtype=int)
np.add.at(out, [0, 2, 0, 4, 2], [10, 20, 30, 40, 5])
out

array([40,  0, 25,  0, 40])

## 5. `einsum` — one call, many operations

Express matrix products, traces, and diagonal sums with compact subscript notation.

In [8]:
M = np.arange(1, 10).reshape(3, 3)
print("diagonal   :", np.einsum("ii->i", M))
print("trace      :", np.einsum("ii->", M))
print("col sums   :", np.einsum("ij->j", M))
print("row sums   :", np.einsum("ij->i", M))

diagonal   : [1 5 9]
trace      : 15
col sums   : [12 15 18]
row sums   : [ 6 15 24]


In [9]:
A = np.random.default_rng(2).random((4, 5))
B = np.random.default_rng(3).random((5, 3))
matmul = np.einsum("ik,kj->ij", A, B)
print("einsum matmul == np.matmul:", np.allclose(matmul, A @ B))

einsum matmul == np.matmul: True


## 6. Smaller dtypes cut memory

`float32` uses half the bytes of `float64` — useful for huge data.

In [10]:
f64 = np.ones((1000, 1000), dtype="float64")
f32 = f64.astype("float32")
print("float64 bytes:", f64.nbytes)
print("float32 bytes:", f32.nbytes)

float64 bytes: 8000000
float32 bytes: 4000000


## Summary

- Prefer **C-order** and slice along the contiguous axis; know your strides.
- Understand **views vs copies** (`np.shares_memory`).
- **Vectorise** — ufuncs, `fromiter`, `ufunc.at`, `einsum`.
- Use smaller dtypes and `memmap` for big data.

Next: **15-analysis-workflow** — a full mini data-analysis project.